In [ ]:
# import os
# os.environ["SHAPE_RESTORE_SHX"] = "YES"

import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

### Calculate population-weighted temperature timeseries by LRZ

In [ ]:
## Load population data
# population data is from US Census Bureau, checked overlap (some counties in Alaska and CT are changed/missing)
pop = pd.read_csv("Population data/co-est2019-alldata.csv", encoding="latin1")
pop = pop[["STATE", "COUNTY", "STNAME", "CTYNAME", "POPESTIMATE2010", "POPESTIMATE2011","POPESTIMATE2012","POPESTIMATE2013","POPESTIMATE2014","POPESTIMATE2015","POPESTIMATE2016","POPESTIMATE2017","POPESTIMATE2018","POPESTIMATE2019"]]

pop2 = pd.read_csv("Population data/co-est2025-alldata.csv", encoding="latin1")
pop2 = pop2[['STATE', 'COUNTY','POPESTIMATE2020', 'POPESTIMATE2021',
       'POPESTIMATE2022', 'POPESTIMATE2023', 'POPESTIMATE2024', 'POPESTIMATE2025']]

pop = pd.merge(pop, pop2, on=["STATE", "COUNTY"], how="left")
pop = pop.rename(columns={"STATE" : "STATEFP", "COUNTY" : "COUNTYFP"})
pop["STATEFP"] = pop["STATEFP"].astype(str).str.zfill(2)
pop["COUNTYFP"] = pop["COUNTYFP"].astype(str).str.zfill(3)


## Load county geographic boundaries
# County boundaries is from US Census Bureau 
counties = (
    gpd.read_file(r"./GIS/cb_2018_us_county_500k/cb_2018_us_county_500k.shp"))


## Load grouped load resource zones
# LRZ boundaries are from PyPSA-USA 
lrz = gpd.read_file(r"./GIS/MISO_grouped_LRZ.shp")

In [ ]:
# Merge population counts into counties dataset
counties = counties.merge(pop, on =["STATEFP", "COUNTYFP"])

# Spatial join: tag each county with the LRZ it falls in (based on centroid)
counties["geometry_orig"] = counties.geometry
counties_pt = counties.copy()
counties_pt["centroid"] = counties_pt.geometry.to_crs("EPSG:5070").centroid.to_crs("EPSG:4326")

counties_pt = counties_pt.set_geometry("centroid")
counties_lrz = gpd.sjoin(counties_pt, lrz[["BA", "geometry"]].to_crs(counties_pt.crs), 
                          how="left", predicate="within")

counties_lrz = counties_lrz.set_geometry("geometry_orig")
counties_lrz = counties_lrz[counties_lrz["BA"].notna()]

centroids = counties_lrz["centroid"]

cx = centroids.x.values  # lons
cy = centroids.y.values  # lats


lats = xr.DataArray(cy, dims="county")
lons = xr.DataArray(cx, dims="county")


## Run for ERA5

In [ ]:
## Process temperature data
# years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
years = [2025]
for year in years:
    ## Load ERA5 data from Turbo (seas-mtcraig)
    ds = xr.open_dataset(rf"/glade/work/rbhandarkar/ERA5/miso_t2m_{year}.nc", chunks="auto")

    county_temps = ds.sel(latitude=lats, longitude=lons, method="nearest")

    pop_weights = counties_lrz[f"POPESTIMATE{year}"].values  # shape: (n_counties,)
    lrz_ids = counties_lrz["BA"].values
    times = ds.valid_time.values

    unique_lrz = lrz["BA"].unique()
    lrz_temp = np.zeros((len(times), len(unique_lrz)))

    county_temps_da = county_temps["t2m"]

    for j, lrz_id in enumerate(unique_lrz):
        mask = lrz_ids == lrz_id
        w = pop_weights[mask]
        t_vals = county_temps_da.isel(county=mask).values   # (time, n_counties_in_lrz)
        
        if w.sum() == 0:
            lrz_temp[:, j] = np.nan
        else:
            lrz_temp[:, j] = np.average(t_vals, weights=w, axis=1)

    # Package as a clean DataFrame or xarray
    result = pd.DataFrame(
        lrz_temp,
        index=pd.to_datetime(times),
        columns=unique_lrz
    )
    result = result -273.15
    result.index.name = "time"

    result.to_csv(rf"C:\Users\ritib\OneDrive - Umich\Research\RA Extremes\Inputs\ERA5\MISO_t2m_{year}.csv")

## Process humidity data
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
for year in years:
    ## Load ERA5 data from Turbo (seas-mtcraig)
    ds = xr.open_dataset(rf"S:\ERA5\ERA5_CONUS\MISO\miso_d2m_{year}.nc", chunks="auto")

    county_hum = ds.sel(latitude=lats, longitude=lons, method="nearest")

    pop_weights = counties_lrz[f"POPESTIMATE{year}"].values  # shape: (n_counties,)
    lrz_ids = counties_lrz["BA"].values
    times = ds.valid_time.values

    unique_lrz = lrz["BA"].unique()
    lrz_hum = np.zeros((len(times), len(unique_lrz)))

    county_hum_da = county_hum["d2m"]

    for j, lrz_id in enumerate(unique_lrz):
        mask = lrz_ids == lrz_id
        w = pop_weights[mask]
        d_vals = county_hum_da.isel(county=mask).values   # (time, n_counties_in_lrz)
        
        if w.sum() == 0:
            lrz_hum[:, j] = np.nan
        else:
            lrz_hum[:, j] = np.average(d_vals, weights=w, axis=1)

    # Package as a clean DataFrame or xarray
    result = pd.DataFrame(
        lrz_hum,
        index=pd.to_datetime(times),
        columns=unique_lrz
    )
    result = result -273.15
    result.index.name = "time"

    result.to_csv(rf"C:\Users\ritib\OneDrive - Umich\Research\RA Extremes\Inputs\ERA5\MISO_d2m_{year}.csv")

## Run for MESACLIP

In [ ]:
## This cell takes too long to run, see process_mesaclip.py instead

## Process temperature data
years = list(np.arange(2030, 2046))
pop_year = 2025
ensembles =  [f"{ens:03}" for ens in  range(1, 11)]

path_map = {"RCP85" : {"002" : "30-2006-2100.002",
            "003" : "31-2006-2100.003",
            "004" : "44-2006-2100.004",
            "005" : "44-2006-2100.005",
            "006" : "46-2006-2100.006",
            "007" : "46-2006-2100.007",
            "008" : "46-2006-2100.008",
            "009" : "46-2006-2100.009",
            "010" : "46-2006-2100.010"
            },
            "RCP60" : {"001" : "44-2006-2100.001",
            "002" : "44-2006-2100.002",
            "003" : "44-2006-2100.003",
            "004" : "46-2006-2100.004",
            "005" : "44-2006-2100.005",
            "006" : "46-2006-2100.006",
            "007" : "46-2006-2100.007",
            "008" : "46-2006-2100.008",
            "009" : "46-2006-2100.009",
            "010" : "46-2006-2100.010"}
            }
rcp_map = {"RCP85" : "d651009",
           "RCP60" : "d651008",
           }


years = np.arange(2015,2025)

for ens in ensembles:
    for rcp in ["RCP60", "RCP85"]:
        if (rcp == "RCP85") & (ens == "001"): continue
        cesm_base = Path(rf"/gdex/data/{rcp_map[rcp]}/b.e13.B{rcp}C5.ne120_t12.cesm-ihesp-hires1.0.{path_map[rcp][ens]}/atm/proc/tseries/hour_6")
        for year in years:
            ## Load MESACLIP data 
            ds = xr.open_dataset(cesm_base / f"b.e13.B{rcp}C5.ne120_t12.cesm-ihesp-hires1.0.{path_map[rcp][ens]}.cam.h2.TREFHT.{year}010100-{year+1}010100.nc")
            
            # MESACLIP data is unstructured, so need to convert grid to points
            lon = ((ds["lon"] + 180) % 360) - 180
            grid = pd.DataFrame({
                "lat": ds["lat"].values,
                "lon": lon.values
            })

            gdf_grid = gpd.GeoDataFrame(
                grid,
                geometry=gpd.points_from_xy(grid.lon, grid.lat),
                crs="EPSG:4326"
            )
            
            # Map each county to point on the grid
            county_points = gpd.GeoDataFrame(
                counties_lrz[["centroid", f"POPESTIMATE{pop_year}", "BA"]],
                geometry=centroids,
                crs="EPSG:4326"
            )

            county_to_grid = gpd.sjoin_nearest(
                county_points.to_crs("EPSG:5070"),
                gdf_grid.to_crs("EPSG:5070"),
                how="left"
            )


            grid_indices = county_to_grid["index_right"].values

            assignment = pd.DataFrame({
                "county_lat": county_points.geometry.y.values,
                "county_lon": county_points.geometry.x.values,
                "population": county_to_grid[f"POPESTIMATE{pop_year}"].values,
                "BA":         county_to_grid["BA"].values,
                "cesm_lat":   grid.iloc[grid_indices]["lat"].values,
                "cesm_lon":   grid.iloc[grid_indices]["lon"].values,
            })
            print("reached assignment point")
            break

            # Map population data to temperature data
            county_temp = ds["TREFHT"].isel(ncol=grid_indices).assign_coords(
                pop=("ncol", np.asarray(county_to_grid[f"POPESTIMATE{pop_year}"])),
                BA=("ncol", np.asarray(county_to_grid["BA"]))
            )

            # Use np.asarray to ensure plain numpy arrays (avoids PyArrow indexing issues)
            pop_weights = np.asarray(county_temp["pop"])
            lrz_ids = np.asarray(county_temp["BA"])
            times = ds.time.values

            unique_lrz = lrz["BA"].unique()
            lrz_temp = np.zeros((len(times), len(unique_lrz)))

            for j, lrz_id in enumerate(unique_lrz):
                mask = lrz_ids == lrz_id
                idx = np.where(mask)[0]
                sub = county_temp.isel(ncol=idx)
                w = xr.DataArray(pop_weights[mask], dims="ncol")
                lrz_temp[:, j] = sub.weighted(w).mean("ncol")  # (time,)


            # Package as a clean DataFrame or xarray
            result = pd.DataFrame(
                lrz_temp,
                index=pd.to_datetime(
                    [str(t) for t in times]
                ),
                columns=unique_lrz
            )
            result = result -273.15
            result.index.name = "time"

            result.to_csv(rf"C:\Users\ritib\OneDrive - Umich\Research\RA Extremes\Inputs\MESACLIP\MISO_{rcp}_{ens}_trefht_{year}.csv")
            ds.close()


In [ ]:
assignment.to_csv("/glade/u/home/rbhandarkar/Inputs/Generator Data/Mapping/county_centroid_to_cesm_mapping.csv")

## Compare ERA5 to MESACLIP

In [ ]:
years = list(np.arange(2015, 2026))
era5_data = pd.DataFrame(columns=["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"])
for year in years:
    temp_ts = pd.read_csv(fr"./ERA5/MISO_t2m_{year}.csv", index_col=0, header=0)
    era5_data =  pd.concat([era5_data, temp_ts], axis=0)

ensembles =  [f"{ens:03}" for ens in  range(1, 11)]
rcps = ["RCP60", "RCP85"]
cesmhr_data = pd.DataFrame(columns=["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"])

for rcp in rcps:
    for ens in ensembles:
        if (rcp == "RCP85") & (ens == "001"): continue
        for year in years:
            temp_ts = pd.read_csv(fr"./MESACLIP/MISO_{rcp}_{ens}_trefht_{year}.csv", index_col=0, header=0)
            cesmhr_data =  pd.concat([cesmhr_data, temp_ts], axis=0)

era5_data["source"] = "ERA5"
cesmhr_data["source"] = "CESM-HR"

combined = pd.concat([era5_data, cesmhr_data])

long_df = combined.reset_index().melt(
    id_vars=["index", "source"], 
    var_name="zone", 
    value_name="temperature"
)
long_df["temperature (°F)"] = long_df["temperature"] * 9/5 + 32

import plotly.express as px

title_map = {"MISO-0001" : "MN & ND", "MISO-0027" : "MI & WI", "MISO-0035": "IA & MO", "MISO-0004": "IL", "MISO-0006":"IN", "MISO-8910": "MS, AR, & LA"}
fig = px.histogram(
    long_df,
    x="temperature (°F)",
    color="source",
    facet_col="zone",
    facet_col_wrap=3,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.5,
    nbins=50
)

fig.update_layout(
    title="Temperature Distribution by Zone: ERA5 vs CESM-HR",
    legend_title="Dataset",
    height=800,
    width=1100,
    template="simple_white"
)

tick_vals = list(range(-20, 130, 10))
tick_text = [str(v) if i % 2 == 0 else "" for i, v in enumerate(tick_vals)]
fig.update_xaxes(
    tickvals=tick_vals,
    ticktext=tick_text,
    showticklabels=True,
)
fig.for_each_annotation(lambda a: a.update(text=title_map.get(a.text.split("=")[-1], a.text)))

# --- std annotations ---
std_df = long_df.groupby(["zone", "source"])["temperature (°F)"].std()

zones_ordered = ["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"]
for i, zone in enumerate(zones_ordered):
    row = i // 3 + 1
    col = i % 3 + 1
    era5_std = std_df.get((zone, "ERA5"), float("nan"))
    cesm_std = std_df.get((zone, "CESM-HR"), float("nan"))
    fig.add_annotation(
        row=row, col=col,
        x=1, y=1,
        xref="x domain", yref="y domain",
        xanchor="right", yanchor="top",
        text=f"ERA5 σ={era5_std:.1f}°F<br>CESM σ={cesm_std:.1f}°F",
        showarrow=False,
        font=dict(size=10),
        bgcolor="rgba(255,255,255,0.75)",
        borderpad=3,
    )

fig.show()


In [ ]:
import plotly.express as px

long_df["month"] = pd.to_datetime(long_df["index"]).dt.strftime("%b")

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = px.box(
    long_df,
    x="month",
    y="temperature (°F)",
    color="source",
    facet_col="zone",
    facet_col_wrap=3,
    points="outliers",
    category_orders={"month": month_order},
)

fig.update_layout(
    title="Temperature Distribution by Month and Zone: ERA5 vs CESM-HR",
    legend_title="Dataset",
    height=700,
    width=1100,
    template="simple_white",
)

fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

return_periods = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 75, 100]

# Recover per-ensemble annual maxima via cumcount on repeated timestamps
long_df["year"] = pd.to_datetime(long_df["index"]).dt.year
long_df["replicate"] = long_df.groupby(["zone", "source", "index"]).cumcount()

ann_max = (
    long_df.groupby(["zone", "source", "year", "replicate"])["temperature (°F)"]
    .max()
    .reset_index()
)

zones = sorted(long_df["zone"].unique())
sources = ["ERA5", "CESM-HR"]
colors = {"ERA5": "#1f77b4", "CESM-HR": "#d62728"}

ncols = 3
nrows = -(-len(zones) // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=list(zones),
    horizontal_spacing=0.08,
    vertical_spacing=0.15,
)

for i, zone in enumerate(zones):
    row = i // ncols + 1
    col = i % ncols + 1

    for source in sources:
        maxima = (
            ann_max[(ann_max["zone"] == zone) & (ann_max["source"] == source)]
            ["temperature (°F)"].values
        )
        n_years = len(maxima)

        # Only include return periods we have enough years to support
        valid_rps = [T for T in return_periods if n_years >= T]
        if not valid_rps:
            continue

        # T=2 → median (q=0.5), T=10 → 90th pct (q=0.9), etc.
        temps = [np.quantile(maxima, 1 - 1/T) for T in valid_rps]
        
        baseline = np.quantile(maxima, 0.5)  # T=2 median
        temps = [t / baseline for t in temps]



        fig.add_trace(
            go.Scatter(
                x=valid_rps, y=temps,
                name=source,
                mode="lines+markers",
                line=dict(color=colors[source]),
                showlegend=(i == 0),
            ),
            row=row, col=col,
        )

fig.update_xaxes(
    title_text="Return Period (years)",
    tickvals=return_periods,
    ticktext=[str(T) for T in return_periods],
)
fig.update_yaxes(title_text="Temperature (°F)")
fig.update_layout(
    title="Annual Max Temperature Return Levels by LRZ: ERA5 vs CESM-HR",
    height=700, width=1100,
    template="simple_white",
)
fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

return_periods = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 75, 100]

long_df["date"] = pd.to_datetime(long_df["index"]).dt.date
long_df["year"] = pd.to_datetime(long_df["index"]).dt.year
long_df["replicate"] = long_df.groupby(["zone", "source", "index"]).cumcount()

# Daily mean, then annual minimum daily mean
daily_mean = (
    long_df.groupby(["zone", "source", "year", "date", "replicate"])["temperature (°F)"]
    .mean()
    .reset_index()
)
ann_min = (
    daily_mean.groupby(["zone", "source", "year", "replicate"])["temperature (°F)"]
    .min()
    .reset_index()
)

zones = sorted(long_df["zone"].unique())
sources = ["ERA5", "CESM-HR"]
colors = {"ERA5": "#1f77b4", "CESM-HR": "#d62728"}

ncols = 3
nrows = -(-len(zones) // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=list(zones),
    horizontal_spacing=0.08,
    vertical_spacing=0.15,
)

for i, zone in enumerate(zones):
    row = i // ncols + 1
    col = i % ncols + 1

    for source in sources:
        minima = (
            ann_min[(ann_min["zone"] == zone) & (ann_min["source"] == source)]
            ["temperature (°F)"].values
        )
        n_years = len(minima)

        valid_rps = [T for T in return_periods if n_years >= T]
        if not valid_rps:
            continue

        # Cold extremes: T=2 → median, T=10 → 10th pct (colder tail)
        temps = [np.quantile(minima, 1 / T) for T in valid_rps]
        # baseline = np.quantile(minima, 0.5)  # T=2 median
        # temps = [t / baseline for t in temps]

        fig.add_trace(
            go.Scatter(
                x=valid_rps, y=temps,
                name=source,
                mode="lines+markers",
                line=dict(color=colors[source]),
                showlegend=(i == 0),
            ),
            row=row, col=col,
        )

fig.update_xaxes(
    title_text="Return Period (years)",
    tickvals=return_periods,
    ticktext=[str(T) for T in return_periods],
)
fig.update_yaxes(title_text="Coldest Day Temp (relative to 2-year return level)")
fig.update_layout(
    title="Annual Coldest Day Return Levels by LRZ: ERA5 vs CESM-HR",
    height=700, width=1100,
    template="simple_white",
)
fig.show()


## Plot against historical loads

In [ ]:
# Read in loads
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
load_data = pd.DataFrame(columns=["LoadResource Zone", "ActualLoad (MWh)"])
for year in years:
    load_ts = pd.read_excel(rf".\MISO historical loads\{year}1231_dfal_hist.xls", index_col = 0, header=5)
    load_ts = load_ts[:-2].drop(columns=['MTLF (MWh)'])
    load_ts =load_ts[~(load_ts.index == "MarketDay")]
    load_ts.index = load_ts.index + pd.to_timedelta(load_ts["HourEnding"]-1, unit="h")
    load_data = pd.concat([load_data, load_ts[["LoadResource Zone", "ActualLoad (MWh)"]]], axis=0)

temp_data = pd.DataFrame(columns=["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"])
for year in years:
    temp_ts = pd.read_csv(fr".\ERA5\MISO_t2m_{year}.csv", index_col=0, header=0)
    temp_data =  pd.concat([temp_data, temp_ts], axis=0)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pivot load data to wide format: index=time, columns=LRZ, values=load
load_wide = load_data.pivot_table(index=load_data.index, columns="LoadResource Zone", values="ActualLoad (MWh)")
load_wide["LRZ8_9_10"] = load_wide["LRZ8_9_10"].fillna(0) + load_wide["LRZ8_9"].fillna(0)

lrz_ids = ["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"]

load_wide= load_wide.drop(columns=["LRZ8_9", "MISO"])
load_wide = load_wide.rename(columns=dict(zip(load_wide.columns, lrz_ids)))
temp_data.columns.name = load_wide.columns.name
temp_data.index = pd.to_datetime(temp_data.index)
temp_data.index = temp_data.index.tz_localize("UTC").tz_convert("Etc/GMT+5")
temp_data.index = pd.to_datetime(temp_data.index).tz_localize(None)
load_wide.index = pd.to_datetime(load_wide.index).tz_localize(None)

n = len(lrz_ids)
cols = 3
rows = (n + cols - 1) // cols


In [ ]:
fig = make_subplots(rows=rows, cols=cols, subplot_titles=list(lrz_ids), shared_xaxes=False)

for i, lrz in enumerate(lrz_ids):
    row, col = divmod(i, cols)
    temp = temp_data[lrz]
    load = load_wide[lrz]

    mask = temp.notna() & load.notna()
    
    fig.add_trace(
        go.Scattergl(
            x=temp[mask],
            y=load[mask],
            mode="markers",
            marker=dict(size=3, opacity=0.4, color=temp[mask], colorscale="RdBu_r"),
            name=lrz,
            hovertemplate="Temp: %{x:.1f}°C<br>Load: %{y:,.0f} MWh<extra></extra>",
        ),
        row=row + 1, col=col + 1
    )
    fig.update_xaxes(
        title_text="Temp (°C)",
        showline=True, linecolor="black", mirror=True,
        row=row + 1, col=col + 1,
        range=[-35, 40],
    )
    fig.update_yaxes(
        title_text="Load (MWh)",
        showline=True, linecolor="black", mirror=True,
        row=row + 1, col=col + 1
    )

fig.update_layout(
    height=300 * rows,
    title="Temperature vs Load by LRZ",
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig.show()

In [ ]:
fig = make_subplots(rows=rows, cols=cols, subplot_titles=list(lrz_ids), shared_xaxes=False)

for i, lrz in enumerate(lrz_ids):
    row, col = divmod(i, cols)
    temp = temp_data[lrz]
    load = load_wide[lrz]

    avg_temp_24h = temp.rolling(window=12, min_periods=12).mean()

    mask = temp.notna() & load.notna()  & avg_temp_24h.notna() 

    fig.add_trace(
        go.Scattergl(
            x=temp[mask],
            y=load[mask],
            mode="markers",
            marker=dict(
                size=3,
                opacity=0.4,
                color=avg_temp_24h[mask],
                colorscale="RdBu_r",
                cmin=-10,
                cmax=25,
                showscale=(i == 0),
                colorbar=dict(
                    title="Avg temp<br>prev 24h (°C)",
                    thickness=15,
                    len=0.5,
                ) if i == 0 else None,
            ),
            name=lrz,
            hovertemplate="Temp: %{x:.1f}°C<br>Load: %{y:,.0f} MWh<br>24h avg: %{marker.color:.1f}°C<extra></extra>",
        ),
        row=row + 1, col=col + 1
    )
    fig.update_xaxes(
        title_text="Temp (°C)",
        showline=True, linecolor="black", mirror=True,
        row=row + 1, col=col + 1,
        range=[-35, 40],
    )
    fig.update_yaxes(
        title_text="Load (MWh)",
        showline=True, linecolor="black", mirror=True,
        row=row + 1, col=col + 1
    )

fig.update_layout(
    height=300 * rows,
    title="Temperature vs Load by LRZ — colored by 24h avg temperature",
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# Each "cell" gets a 2x2 block: [scatter, y-hist] / [x-hist, empty]
# We'll use specs to control sizing via column_widths / row_heights per block

# Build a grid where each LRZ occupies a 2-row × 2-col block
# Main scatter:  (block_row,   block_col)
# X histogram:   (block_row+1, block_col)
# Y histogram:   (block_row,   block_col+1)

n = len(lrz_ids)
cols = 3  # number of LRZ columns
rows = math.ceil(n / cols)

# Total subplot grid dimensions
total_rows = rows * 2
total_cols = cols * 2

# Alternate row heights: [scatter_height, hist_height, scatter_height, hist_height, ...]
row_heights = []
for _ in range(rows):
    row_heights += [0.8, 0.2]

col_widths = []
for _ in range(cols):
    col_widths += [0.8, 0.2]

# Build specs: all are xy plots; corner cells (x-hist row, y-hist col) are empty
specs = []
for r in range(total_rows):
    spec_row = []
    for c in range(total_cols):
        is_xhist_row = (r % 2 == 1)
        is_yhist_col = (c % 2 == 1)
        if is_xhist_row and is_yhist_col:
            spec_row.append(None)  # dead corner cell
        else:
            spec_row.append({"type": "xy"})
    specs.append(spec_row)

fig = make_subplots(
    rows=total_rows,
    cols=total_cols,
    specs=specs,
    row_heights=row_heights,
    column_widths=col_widths,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

for i, lrz in enumerate(lrz_ids):
    lrz_row, lrz_col = divmod(i, cols)

    # Subplot grid positions (1-indexed)
    scatter_row = lrz_row * 2 + 1
    scatter_col = lrz_col * 2 + 1
    xhist_row   = lrz_row * 2 + 2
    xhist_col   = lrz_col * 2 + 1
    yhist_row   = lrz_row * 2 + 1
    yhist_col   = lrz_col * 2 + 2

    temp = temp_data[lrz]
    load = load_wide[lrz]
    mask = temp.notna() & load.notna() & (temp.index.hour == 16)  & (temp.index.dayofweek < 5)
    x = temp[mask]
    y = load[mask]

    # --- Scatter ---
    fig.add_trace(
        go.Scattergl(
            x=x, y=y,
            mode="markers",
            marker=dict(size=3, opacity=0.4, color=x, colorscale="RdBu_r"),
            name=lrz,
            hovertemplate="Temp: %{x:.1f}°C<br>Load: %{y:,.0f} MWh<extra></extra>",
        ),
        row=scatter_row, col=scatter_col,
    )

    # --- X histogram (temperature, below scatter) ---
    fig.add_trace(
        go.Histogram(
            x=x,
            nbinsx=40,
            marker_color="steelblue",
            opacity=0.7,
            showlegend=False,
            hovertemplate="Temp: %{x:.1f}°C<br>Count: %{y}<extra></extra>",
        ),
        row=xhist_row, col=xhist_col,
    )

    # --- Y histogram (load, right of scatter) ---
    fig.add_trace(
        go.Histogram(
            y=y,
            nbinsy=40,
            marker_color="steelblue",
            opacity=0.7,
            showlegend=False,
            hovertemplate="Load: %{y:,.0f} MWh<br>Count: %{x}<extra></extra>",
        ),
        row=yhist_row, col=yhist_col,
    )

    # --- Axis styling ---
    fig.update_xaxes(
        showline=True, linecolor="black", mirror=True,
        range=[-35, 40],
        row=scatter_row, col=scatter_col,
    )
    fig.update_yaxes(
        title_text="Load (MWh)",
        showline=True, linecolor="black", mirror=True,
        row=scatter_row, col=scatter_col,
    )
    # X-hist axes
    fig.update_xaxes(
        title_text="Temp (°C)",
        range=[-35, 40],
        showline=False,  
        showticklabels=False,
        row=xhist_row, col=xhist_col,
    )
    fig.update_yaxes(
        autorange="reversed",  # flips histogram upside down
        showticklabels=False,
        row=xhist_row, col=xhist_col,
    )
    # Y-hist axes
    fig.update_xaxes(
        showticklabels=False,
        row=yhist_row, col=yhist_col,
    )
    fig.update_yaxes(
        showticklabels=False,
        row=yhist_row, col=yhist_col,
    )

# Subplot titles — manually annotate since make_subplots titles don't align well with 2x2 blocks
for i, lrz in enumerate(lrz_ids):
    lrz_row, lrz_col = divmod(i, cols)
    scatter_row = lrz_row * 2 + 1
    scatter_col = lrz_col * 2 + 1
    fig.update_annotations(
        # make_subplots won't have titles here, add them via layout annotations below
    )

# Add LRZ titles as layout annotations above each scatter plot
annotations = list(fig.layout.annotations)
for i, lrz in enumerate(lrz_ids):
    lrz_row, lrz_col = divmod(i, cols)
    scatter_row = lrz_row * 2 + 1
    scatter_col = lrz_col * 2 + 1
    axis_key = f"x{(scatter_row - 1) * total_cols + scatter_col}" if (scatter_row > 1 or scatter_col > 1) else "x"
    annotations.append(dict(
        text=str(lrz),
        xref=f"x{(scatter_row - 1) * total_cols + scatter_col} domain" if i > 0 else "x domain",
        yref=f"y{(scatter_row - 1) * total_cols + scatter_col} domain" if i > 0 else "y domain",
        x=0.5, y=1.08,
        showarrow=False,
        font=dict(size=12),
        xanchor="center",
    ))

fig.update_layout(
    height=350 * rows,
    title="Temperature vs Load by LRZ",
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
    bargap=0.05,
    annotations=annotations,
)

fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
z = "0001"

# Load (primary y-axis)
fig.add_trace(go.Scattergl(
    x=load_wide.index,
    y=load_wide[f"MISO-{z}"],
    mode="lines",
    line=dict(color="black", width=0.5),
    name=f"LRZ {z}",
    yaxis="y"
))

# Temperature (secondary y-axis)
fig.add_trace(go.Scattergl(
    x=temp_data.index,
    y=temp_data[f"MISO-{z}"],
    mode="lines",
    line=dict(color="red", width=0.5),
    name="Temperature",
    yaxis="y2"
))

fig.update_layout(
    title=f"Load vs Temperature — LRZ {z}",
    xaxis=dict(
        title="Time",
        showline=True,
        linecolor="black",
        mirror=True
    ),
    yaxis=dict(
        title="Load (MWh)",
        showline=True,
        linecolor="black",
        mirror=True
    ),
    yaxis2=dict(
        title="Temperature",
        overlaying="y",   # share same x-axis
        side="right",
        showline=True,
        linecolor="black",
        mirror=True
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf

z = "0035"
nlags = 24 * 7 * 2  # two weeks of hourly lags

def make_acf_trace(series, nlags, color):
    acf_vals, confint = acf(series.dropna(), nlags=nlags, alpha=0.05, fft=True)
    lags = np.arange(len(acf_vals))

    traces = []

    # Confidence band centered on zero
    traces.append(go.Scatter(
        x=np.concatenate([lags, lags[::-1]]),
        y=np.concatenate([confint[:, 1] - acf_vals, (confint[:, 0] - acf_vals)[::-1]]),
        fill="toself",
        fillcolor=color,
        opacity=0.15,
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    ))

    # Vertical stems
    for lag, val in zip(lags, acf_vals):
        traces.append(go.Scatter(
            x=[lag, lag],
            y=[0, val],
            mode="lines",
            line=dict(color=color, width=0.8),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Dot markers
    traces.append(go.Scatter(
        x=lags,
        y=acf_vals,
        mode="markers",
        marker=dict(color=color, size=3),
        showlegend=False,
        hovertemplate="Lag %{x}<br>ACF: %{y:.3f}<extra></extra>",
    ))

    return traces


fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[f"ACF — Load LRZ {z}", "ACF — Temperature"],
    vertical_spacing=0.12,
)

for trace in make_acf_trace(load_wide[f"MISO-{z}"], nlags, color="black"):
    fig.add_trace(trace, row=1, col=1)

for trace in make_acf_trace(temp_data[f"MISO-{z}"], nlags, color="steelblue"):
    fig.add_trace(trace, row=2, col=1)

fig.update_layout(
    height=700,
    paper_bgcolor="white",
    plot_bgcolor="white",
    title_text=f"Autocorrelation functions — LRZ {z}",
)

for row in [1, 2]:
    fig.update_xaxes(title_text="Lag (hours)", showline=True, linecolor="black", mirror=True, row=row, col=1)
    fig.update_yaxes(title_text="Autocorrelation", showline=True, linecolor="black", mirror=True, zeroline=True, zerolinecolor="black", zerolinewidth=0.5, row=row, col=1)

fig.show()

In [ ]:
from statsmodels.tsa.stattools import pacf, ccf
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

z = "0027"
lrz = f"MISO-{z}"

mask = temp_data[lrz].notna() & load_wide[lrz].notna() & (temp_data.index.month < 10)  & (temp_data.index.month > 3)
x = temp_data[lrz][mask]
y = load_wide[lrz][mask]

nlags = 48

pacf_vals, pacf_confint = pacf(y, nlags=nlags, alpha=0.05)
ccf_vals = ccf(x, y, nlags=nlags)

n = len(x)
ci = 1.96 / np.sqrt(n)


def make_stem_traces(lags, vals, ci_lower, ci_upper, color, hover_label):
    traces = []

    traces.append(go.Scatter(
        x=np.concatenate([lags, lags[::-1]]),
        y=np.concatenate([ci_upper, ci_lower[::-1]]),
        fill="toself",
        fillcolor=color,
        opacity=0.15,
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    ))

    for lag, val in zip(lags, vals):
        traces.append(go.Scatter(
            x=[lag, lag],
            y=[0, val],
            mode="lines",
            line=dict(color=color, width=0.8),
            showlegend=False,
            hoverinfo="skip",
        ))

    traces.append(go.Scatter(
        x=lags,
        y=vals,
        mode="markers",
        marker=dict(color=color, size=3),
        showlegend=False,
        hovertemplate=f"Lag %{{x}}h<br>{hover_label}: %{{y:.3f}}<extra></extra>",
    ))

    return traces


pacf_lags = np.arange(len(pacf_vals))
pacf_ci_upper = pacf_confint[:, 1] - pacf_vals
pacf_ci_lower = pacf_confint[:, 0] - pacf_vals

ccf_lags = np.arange(len(ccf_vals))
ccf_ci_upper = np.full(len(ccf_lags), ci)
ccf_ci_lower = np.full(len(ccf_lags), -ci)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        f"PACF — Load LRZ {z}",
        f"CCF — Temperature → Load LRZ {z}",
    ],
    vertical_spacing=0.14,
)

for trace in make_stem_traces(pacf_lags, pacf_vals, pacf_ci_lower, pacf_ci_upper, "black", "PACF"):
    fig.add_trace(trace, row=1, col=1)

for trace in make_stem_traces(ccf_lags, ccf_vals, ccf_ci_lower, ccf_ci_upper, "steelblue", "CCF"):
    fig.add_trace(trace, row=2, col=1)

fig.update_layout(
    height=700,
    paper_bgcolor="white",
    plot_bgcolor="white",
    title_text=f"PACF and CCF — LRZ {z}",
)

for row in [1, 2]:
    fig.update_xaxes(
        title_text="Lag (hours)",
        showline=True, linecolor="black", mirror=True,
        tickmode="linear", dtick=6,
        row=row, col=1,
    )
    fig.update_yaxes(
        title_text="Correlation",
        showline=True, linecolor="black", mirror=True,
        zeroline=True, zerolinecolor="black", zerolinewidth=0.5,
        row=row, col=1,
    )

fig.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

lrz = "MISO-0027"
temp = temp_data[lrz]
load = load_wide[lrz]

# build lag features
df = pd.DataFrame({"load": load, "temp": temp})
df["hour"] = df.index.hour
df["dow"] = df.index.dayofweek

# model 1: no temperature lags
for h in range(1, 4):
    df[f"load_lag{h}"] = df["load"].shift(h)

# model 2: add temperature lags
for h in range(1, 13):
    df[f"temp_lag{h}"] = df["temp"].shift(h)

df = df.dropna()

# split
n = len(df)
train = df.iloc[:int(n * 0.8)]
test  = df.iloc[int(n * 0.8):]

features_base = ["temp", "hour", "dow", "load_lag1", "load_lag2", "load_lag3"]
features_lags = features_base + [f"temp_lag{h}" for h in range(1, 13)]

for name, features in [("Base (no temp lags)", features_base), ("With temp lags", features_lags)]:
    m = LinearRegression().fit(train[features], train["load"])
    preds = m.predict(test[features])
    rmse = np.sqrt(mean_squared_error(test["load"], preds))
    print(f"{name}: RMSE = {rmse:,.1f}")

In [ ]:
import statsmodels.api as sm
import numpy as np

lag_names = ["temp"] + [f"temp_lag{h}" for h in range(1, 13)]

# build dummies manually — much more memory efficient
hour_dummies = pd.get_dummies(df["hour"], prefix="hour", drop_first=True).astype(int)
dow_dummies  = pd.get_dummies(df["dow"],  prefix="dow",  drop_first=True).astype(int)

X = pd.concat([df[lag_names], hour_dummies, dow_dummies], axis=1).astype(float)
X = sm.add_constant(X)
y = df["load"].astype(float)

model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 24})
print(model.summary())

ci = model.conf_int()
coefs    = [model.params[n] for n in lag_names]
ci_upper = [ci.loc[n, 1] for n in lag_names]
ci_lower = [ci.loc[n, 0] for n in lag_names]
lags      = list(range(13))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lags + lags[::-1],
    y=ci_upper + ci_lower[::-1],
    fill="toself",
    fillcolor="black",
    opacity=0.15,
    line=dict(width=0),
    showlegend=False,
))

fig.add_trace(go.Scatter(
    x=lags,
    y=coefs,
    mode="lines+markers",
    line=dict(color="black", width=1.5),
    marker=dict(size=5),
    name="Coefficient",
    hovertemplate="Lag %{x}h<br>%{y:,.1f} MWh per °C<extra></extra>",
))

fig.add_hline(y=0, line_width=0.8, line_color="black", opacity=0.4)

fig.update_layout(
    title="Impulse response — load response to temperature shock",
    xaxis_title="Lag (hours)",
    yaxis_title="Change in load (MWh) per 1°C",
    xaxis=dict(tickmode="linear", dtick=1, showline=True, linecolor="black", mirror=True),
    yaxis=dict(showline=True, linecolor="black", mirror=True, zeroline=False),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.show()